# DSAA — DBSCAN Parameter Tuning (v3)

**Purpose.** The current DBSCAN configuration reaches a silhouette score of only **0.2387**, below the v2 result of 0.3587. The main notebook flags this itself:

> `WARNING: Weak separation — consider tuning eps or using cosine metric.`

This notebook sweeps both `eps` and `metric` and reports the best configurations.

**Why cosine.** Each anomaly fingerprint is L1-normalised — every row sums to 1.0, so the fingerprints are compositional (proportion) vectors. Cosine distance is the appropriate metric for compositional data; Euclidean distance on the simplex is not. This is a standard, defensible methodological argument.

**This notebook writes nothing to Google Drive.** It only reads `fingerprints.npz` and prints results. All existing v3 artefacts are left untouched. Apply the chosen parameters in the main notebook to regenerate outputs.

**Runtime:** approximately 5 minutes.

In [ ]:
# ============================================================
# CELL 1: Mount Drive and load the saved fingerprints
# ============================================================
import numpy as np
from google.colab import drive
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score

drive.mount('/content/drive')

dsaa_dir = '/content/drive/MyDrive/DeepSentinel/DeepSentinel_DSAA_v3'

d = np.load(f'{dsaa_dir}/fingerprints.npz', allow_pickle=True)
print('Arrays in npz:', list(d.keys()))

fingerprints   = d['fingerprints']
cluster_labels = d['cluster_labels']   # existing labels, for comparison

n_cur   = len(set(cluster_labels[cluster_labels != -1]))
noise_c = int((cluster_labels == -1).sum())

print(f'\nFingerprints shape : {fingerprints.shape}   '
      f'(13 Signal-1 + 16 Signal-2 = 29 dims)')
print(f'Row sums (first 5) : {fingerprints[:5].sum(axis=1).round(4)}')
print(f'\nCurrent config     : euclidean, eps=0.1075 -> {n_cur} clusters, '
      f'noise={noise_c}, silhouette=0.2387')

In [ ]:
# ============================================================
# CELL 2: Sweep eps x metric
# ============================================================
# Do not select on silhouette alone. A configuration that keeps a few tight
# clusters and discards everything else as noise scores well but is useless.
# The noise fraction and the largest-cluster share are printed alongside so
# the trade-off stays visible.
# ============================================================
MIN_SAMPLES = 10          # same value the main notebook uses
N = len(fingerprints)

rows = []
for metric, grid in [('euclidean', np.arange(0.05, 0.31, 0.01)),
                     ('cosine',    np.arange(0.02, 0.31, 0.01))]:
    for eps in grid:
        lab  = DBSCAN(eps=float(eps), min_samples=MIN_SAMPLES,
                      metric=metric, n_jobs=-1).fit_predict(fingerprints)
        keep = lab != -1
        k    = len(set(lab[keep]))

        # keep only usable solutions
        if k < 4 or k > 30 or keep.sum() < 0.5 * N:
            continue

        sizes   = np.bincount(lab[keep])
        biggest = sizes.max() / N
        if biggest > 0.85:            # one dominant blob is not a typology set
            continue

        s = silhouette_score(fingerprints[keep], lab[keep], metric=metric,
                             sample_size=3000, random_state=42)
        rows.append((float(s), metric, float(eps), int(k),
                     int((~keep).sum()), float(biggest)))

rows.sort(reverse=True)

print(f"{'#':>3} {'silhouette':>11} {'metric':<10} {'eps':>6} "
      f"{'clusters':>9} {'noise':>7} {'noise%':>7} {'largest%':>9}")
print('-' * 72)
for i, (s, m, e, k, n, big) in enumerate(rows[:15]):
    print(f'{i:>3} {s:>11.4f} {m:<10} {e:>6.3f} {k:>9} '
          f'{n:>7} {n/N*100:>6.1f}% {big*100:>8.1f}%')

print(f'\nBaseline — v3 current : 0.2387  euclidean  eps=0.108  12 clusters')
print(f'Baseline — v2         : 0.3587  euclidean  eps=0.118  19 clusters')
print(f'\n{len(rows)} usable configurations found.')

In [ ]:
# ============================================================
# CELL 3: Preview the selected configuration
# ============================================================
# Set CHOICE to the '#' column of the row you want from the table above.
# 0 is the highest silhouette. Change it if another row trades off better.
# ============================================================
CHOICE = 0

BEST_SIL, BEST_METRIC, BEST_EPS, BEST_K, BEST_NOISE, _ = rows[CHOICE]

lab  = DBSCAN(eps=BEST_EPS, min_samples=MIN_SAMPLES,
              metric=BEST_METRIC, n_jobs=-1).fit_predict(fingerprints)
keep = lab != -1

print('=' * 62)
print(f'Selected : metric={BEST_METRIC}  eps={BEST_EPS:.4f}  '
      f'min_samples={MIN_SAMPLES}')
print('=' * 62)
print(f'  Clusters   : {BEST_K}      (current 12 | v2 19)')
print(f'  Noise      : {BEST_NOISE} ({BEST_NOISE/N*100:.1f}%)   (current 505 / 6.1%)')
print(f'  Silhouette : {BEST_SIL:.4f}   (current 0.2387 | v2 0.3587)')

verdict = ('PASS  - better than v2'          if BEST_SIL > 0.3587 else
           'PARTIAL - better than current v3, below v2' if BEST_SIL > 0.2387 else
           'FAIL  - no improvement')
print(f'  Verdict    : {verdict}')

print('\nCluster sizes:')
for cid in sorted(set(lab)):
    c = int((lab == cid).sum())
    name = 'noise' if cid == -1 else f'cluster {cid}'
    print(f'  {name:<12} {c:>6} ({c/N*100:>5.1f}%)')

print('\n' + '=' * 62)
print('Next: apply these two lines in CELL 7 of the main DSAA notebook')
print('=' * 62)
print(f"EPS = {BEST_EPS:.4f}")
print(f"dbscan = DBSCAN(eps=EPS, min_samples=MIN_SAMPLES, "
      f"metric='{BEST_METRIC}', n_jobs=-1)")

## Next steps

1. Copy the two lines printed by CELL 3.
2. In **`DeepSentinel_DSAA_Framework_V3.ipynb`**, open **CELL 7** and change:
   - `EPS = suggested_eps` to the new value
   - `metric='euclidean'` to the new metric
3. In **CELL 13**, the metric is hardcoded as `'metric': 'euclidean'`. Change it to `'metric': dbscan.metric` so the saved `dbscan_config.json` records the metric actually used rather than a stale literal.
4. Run the main notebook from CELL 1 through CELL 14 (about 40 minutes). The typology table, radar chart and dashboard are all regenerated from the new clustering.

**Do not run CELL 7 alone and stop.** Cells 8 to 14 would still hold the previous labels, leaving the saved artefacts inconsistent with each other.

### Reading the table

| Condition | Meaning |
| --- | --- |
| silhouette > 0.3587 | Better than v2 — take it |
| noise% > 20 | Too much fraud discarded as noise |
| largest% > 60 | One dominant blob, not a set of typologies |

CELL 2 already filters to 4–30 clusters, noise below 50% and largest cluster below 85%, but the final choice is a judgement call.